# 📖 Notebook 3 — Cache Headers & Invalidation

A CDN is only as good as its *freshness* story. If edges keep serving the
old logo after you've replaced it on the origin, your users see a stale
brand. If edges pound the origin for every request, your CDN isn't doing
its job. The balance is controlled by **HTTP cache headers**.

In this notebook we'll look at the three tools origins use to control edge
caching:

| Header          | What it does                                                  |
|-----------------|---------------------------------------------------------------|
| `Cache-Control` | How long content may be reused. `max-age=N` = fresh for N seconds. |
| `ETag`          | A fingerprint of the content. Lets caches ask *"still the same?"*.  |
| **Invalidation**    | Force-remove content from the cache when TTL isn't short enough. |

> 🧀 Analogy: `Cache-Control: max-age` is the **"use by"** date on a block
> of cheese. `ETag` is a barcode that lets the fridge check if the cheese
> is the same block you remember. Invalidation is throwing the cheese out
> early because the recall notice just arrived.


## 🛠️ Setup

**Step 1 — Start the lab infrastructure** (from the `01-foundations/cdn/` directory):

```bash
docker compose up -d --build
```

This starts three containers:

| Service | Role                | URL                    |
|---------|---------------------|------------------------|
| origin  | Slow FastAPI server | http://localhost:8000  |
| edge1   | nginx edge POP #1   | http://localhost:8081  |
| edge2   | nginx edge POP #2   | http://localhost:8082  |

**Step 2 — Install Python deps**:

```bash
uv sync
```

**Step 3 — Select the kernel**: in VS Code, click the kernel picker in the
top-right of this notebook and choose the `.venv` interpreter. If it doesn't
show up, reload the window (`Cmd+Shift+P` → *Reload Window*) and try again.


In [ ]:
import time
import statistics
import httpx

ORIGIN = "http://localhost:8000"
EDGE1  = "http://localhost:8081"
EDGE2  = "http://localhost:8082"

def timed_get(url: str, **kwargs) -> tuple[float, httpx.Response]:
    """Return (elapsed_ms, response) for a single GET."""
    t0 = time.perf_counter()
    r = httpx.get(url, timeout=10.0, **kwargs)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return elapsed_ms, r

def show(url: str, elapsed_ms: float, r: httpx.Response) -> None:
    cache = r.headers.get("x-cache-status", "-")
    edge  = r.headers.get("x-edge-server", "-")
    print(f"{url:35s}  {elapsed_ms:7.1f} ms  cache={cache:8s}  edge={edge}")

# Helper — print the cache-related headers of a response.
def show_headers(r):
    for h in ["cache-control", "etag", "x-cache-status", "x-edge-server", "age"]:
        v = r.headers.get(h)
        if v is not None:
            print(f"  {h:16s}: {v}")


## 1️⃣ Cache-Control: the TTL contract

Our origin sets `Cache-Control: public, max-age=30` on every asset. That's
the origin telling every cache in the world:

> *"You may reuse this response for 30 seconds. After that, check with me again."*

Let's inspect the headers coming back through edge1:

In [ ]:
ms, r = timed_get(f"{EDGE1}/assets/logo.svg?t=ttl-demo")
print(f"status={r.status_code}  elapsed={ms:.1f} ms")
show_headers(r)

## 2️⃣ Watching a cache entry live and die

With `max-age=30`, we can literally watch an entry expire. We'll hit the
same edge URL every few seconds and print the cache status and response
time. You should see:

- `MISS` → origin fetch (~500 ms)
- several `HIT`s (~2 ms)
- eventually an `EXPIRED` (or another MISS) once 30 s has passed

In [ ]:
url = f"{EDGE1}/assets/logo.svg?t=ttl-watch"

start = time.time()
while time.time() - start < 40:
    ms, r = timed_get(url)
    age = r.headers.get("age", "?")
    print(f"t+{time.time()-start:4.1f}s  {ms:6.1f} ms  "
          f"cache={r.headers.get('x-cache-status'):8s}  age={age}")
    time.sleep(4)

## 3️⃣ Stale edges: when the origin changes but the edge doesn't

This is the scary one. If the origin updates an asset but the edge still
holds a pre-expiration copy, users get the old version until the TTL runs
out. Let's force that situation by:

1. Requesting the file through an edge so it gets cached.
2. Changing the file on the host (which means the origin now serves v2).
3. Asking the edge again — it still returns v1!

In [ ]:
import pathlib

asset = pathlib.Path("../origin/assets/stale_demo.txt")
asset.write_text("VERSION 1 — fresh from origin\n")

url = f"{EDGE1}/assets/stale_demo.txt?t=stale-demo"

# (1) Prime the edge cache with v1
ms, r = timed_get(url)
print("after first fetch (edge caches v1):")
print("   body:", r.text.strip())
show_headers(r)

# (2) Update the file — origin will now serve v2 to anyone who asks
asset.write_text("VERSION 2 — updated on origin\n")

# (3) Ask the edge again. It's still within max-age=30, so HIT with v1.
ms, r = timed_get(url)
print("\nsecond fetch through edge1 (still within TTL):")
print("   body:", r.text.strip(), "  <-- stale!")
show_headers(r)

# Sanity check: origin really does have v2 right now.
ms, r_origin = timed_get(f"{ORIGIN}/assets/stale_demo.txt")
print("\ndirect origin fetch:")
print("   body:", r_origin.text.strip(), "  <-- the truth")

☝️ edge1 is **lying** to our users — not on purpose, just because
its TTL hasn't run out yet. This is the fundamental trade-off of caching:
**freshness vs. speed**. Shorter TTL = fresher data but more origin traffic.
Longer TTL = fewer origin hits but staler content.

## 4️⃣ Two ways to fix stale content

### Option A — Cache busting with a versioned URL

Instead of invalidating the cache, change the **URL** when content changes:
`logo.svg?v=2` is a different cache key from `logo.svg?v=1`, so the edge
treats it as brand-new content and fetches it fresh. This is why production
sites hash-fingerprint their asset filenames (`app.8f3c1.js`).

In [ ]:
# A versioned URL bypasses the stale cache entry entirely.
url_v2 = f"{EDGE1}/assets/stale_demo.txt?v=2"
ms, r = timed_get(url_v2)
print("versioned URL (new cache key):")
print("   body:", r.text.strip())
show_headers(r)

### Option B — Purge (invalidate) the cache

The other approach is to reach into the edge and delete the cached entry.
Big CDNs expose an API for this (`PURGE /path`, or a dashboard button).

We don't have that API, but we *do* have something better for teaching:
nginx stores its cache on disk inside a docker volume. We can clear it by
emptying that volume. In production you'd use the CDN's purge API instead
— the principle is the same.

In [ ]:
import subprocess

# Delete everything in edge1's on-disk cache (requires docker on PATH).
res = subprocess.run(
    ["docker", "exec", "cdn-edge1", "sh", "-c", "rm -rf /var/cache/nginx/edge/*"],
    capture_output=True, text=True,
)
print("purge exit code:", res.returncode)
if res.stderr:
    print(res.stderr)

# Retry the *original* URL — the edge has no entry, so it fetches v2 fresh.
ms, r = timed_get(f"{EDGE1}/assets/stale_demo.txt?t=stale-demo")
print("\nafter purge — same URL as before:")
print("   body:", r.text.strip())
show_headers(r)

## 5️⃣ ETag & conditional requests

`ETag` is a small fingerprint (we used md5 of the file bytes) that lets a
cache ask the origin: *"I have a version tagged `abc123`. Is it still good?"*

If yes, the origin replies with `304 Not Modified` and **no body** — tiny,
fast, no bandwidth. If no, it sends the new content. nginx can be configured
to do this automatically with `proxy_cache_revalidate on`, but even without
that, clients (browsers) use ETags all the time.

Let's simulate a client sending `If-None-Match` directly to the origin:

In [ ]:
# Get the current ETag
_, r = timed_get(f"{ORIGIN}/assets/logo.svg")
etag = r.headers.get("etag")
print("origin ETag:", etag)

# Now send a conditional request. Expect 304 Not Modified.
ms, r2 = timed_get(f"{ORIGIN}/assets/logo.svg", headers={"If-None-Match": etag})
print(f"\nconditional GET -> status={r2.status_code}  elapsed={ms:.1f} ms  body_bytes={len(r2.content)}")

Note two things:

- Status is **304** — "you already have the latest".
- The body is **empty** (0 bytes). We saved bandwidth, even though we
  still paid the ~500 ms origin delay. In a real deployment the origin's
  304 check would be much faster than rebuilding the full response.

## 📦 Recap

| Problem                        | Tool                                |
|--------------------------------|-------------------------------------|
| "How long can the edge reuse this?" | `Cache-Control: max-age=N`       |
| "Is the copy I have still current?" | `ETag` + `If-None-Match` → 304   |
| "I just changed v1 → v2 NOW"        | Versioned URLs (easy) or purge API (instant) |

In a real CDN:

- **Static, hash-named assets** (`main.8f3c1.js`) get huge TTLs (1 year)
  because the URL itself changes when the content changes.
- **HTML pages** get short TTLs (seconds) or `no-cache` + ETag.
- **User-specific pages** set `Cache-Control: private` so edges never cache
  them at all.

You now have all the mental machinery to read any CDN config and predict
what will (and won't) be served stale. 🎉